# Занятие 24.10.2026

## Тема: алгоритмическая и пространственная сложность

Два вопроса, которые можно задать про любой код:

1. Насколько он замедлится, если данных станет больше?
2. Сколько лишней памяти он при этом займёт?

Пойдём так: сначала руками замерим время и память, увидим закономерности, и только потом введём для них короткую запись - O-нотацию.

# 1. Замеряем время

## 1.1. Как замерить время работы кода

`time.perf_counter()` возвращает число секунд, прошедших с какого-то произвольного момента. Само это число бесполезно, а вот разность двух вызовов - это время работы кода между ними.

In [ ]:
import time

start = time.perf_counter()

total = 0
for i in range(1_000_000):
    total += i

elapsed = time.perf_counter() - start
print("сумма:", total)
print(f"время: {elapsed:.3f} с")

## 1.2. Главный вопрос: а если данных станет в 10 раз больше?

Одно измерение не говорит ни о чём. Интересно другое: как время меняется, когда растёт объём данных. Запустим один и тот же код на разных `n`.

In [ ]:
def sum_all(n):
    total = 0
    for i in range(n):
        total += i
    return total


for n in [100_000, 1_000_000, 10_000_000]:
    start = time.perf_counter()
    sum_all(n)
    print(f"n = {n:>10}   время = {time.perf_counter() - start:.3f} с")

Данных в 10 раз больше - время примерно в 10 раз больше. Такой рост называют **линейным**: во сколько раз выросли данные, во столько же раз вырос расход времени.

## 1.3. А теперь цикл в цикле

Функция ниже для каждого числа пробегает весь список ещё раз. Сравним рост при удвоении `n`.

In [ ]:
def count_zero_pairs(items):
    count = 0
    for a in items:
        for b in items:
            if a + b == 0:
                count += 1
    return count


for n in [1000, 2000, 4000]:
    data = list(range(-n // 2, n // 2))
    start = time.perf_counter()
    count_zero_pairs(data)
    print(f"n = {n:>5}   время = {time.perf_counter() - start:.3f} с")

Данных в 2 раза больше - время примерно в 4 раза больше. Данных в 4 раза больше - время в 16 раз. Это **квадратичный** рост, и он опасен: на `n = 100 000` такой код будет считать часами.

## 1.4. Секунды - плохая мера, считаем шаги

Секунды зависят от компьютера, загрузки системы и версии Python. У вас числа выше получились не такие, как у меня. Поэтому алгоритмы сравнивают не в секундах, а в **шагах** - сколько раз выполнится тело цикла.

In [ ]:
def steps_linear(n):
    steps = 0
    for i in range(n):
        steps += 1
    return steps


def steps_quadratic(n):
    steps = 0
    for i in range(n):
        for j in range(n):
            steps += 1
    return steps


for n in [10, 100, 1000]:
    print(f"n = {n:>5}   один цикл: {steps_linear(n):>10}   цикл в цикле: {steps_quadratic(n):>10}")

Шаги не зависят от железа: один цикл делает ровно `n` шагов, цикл в цикле - `n * n`. Это и есть то, что мы хотим измерять.

# 2. O-нотация

## 2.1. Короткая запись наблюдения

Вместо «шагов примерно `n`» пишут **O(n)**, вместо «примерно `n * n`» - **O(n^2)**, вместо «число шагов не зависит от `n`» - **O(1)**. Читается «порядка n».

O-нотация отвечает не на вопрос «сколько именно операций», а на вопрос «как растёт время, когда растут данные».

In [ ]:
print(f"{'n':>8} {'O(1)':>8} {'O(n)':>10} {'O(n^2)':>14}")
for n in [10, 100, 1000, 10_000]:
    print(f"{n:>8} {1:>8} {n:>10} {n * n:>14}")

## 2.2. Константы и младшие слагаемые отбрасываем

Код, который делает `3 * n` шагов, и код, который делает `n` шагов, - оба O(n). Разница в 3 раза важна на практике, но не меняет характер роста: удвоили данные - оба замедлились вдвое.

Слагаемые тоже отбрасывают, оставляя самое быстро растущее: `n * n + n` - это O(n^2).

In [ ]:
for n in [10, 100, 1000, 100_000]:
    total = n * n + n
    print(f"n = {n:>7}   n*n + n = {total:>14}   доля слагаемого n = {n / total * 100:.4f}%")

Чем больше `n`, тем меньше значит всё, кроме главного слагаемого. Поэтому его одного и записывают.

## 2.3. Основные классы сложности

| Запись | Как растёт | Пример |
|---|---|---|
| O(1) | не растёт вообще | `items[5]`, `len(items)`, `x in some_set` |
| O(log n) | очень медленно | каждый шаг отбрасывает половину данных |
| O(n) | пропорционально | один проход циклом, `sum`, `max`, `x in items` |
| O(n log n) | чуть хуже, чем пропорционально | `sorted(items)` |
| O(n^2) | катастрофически | цикл в цикле |

Сколько это шагов при `n = 1 000 000`: O(1) - один шаг, O(log n) - около 20, O(n) - миллион, O(n log n) - 20 миллионов, O(n^2) - триллион (это примерно 11 дней на обычном ноутбуке).

In [ ]:
# Откуда берётся O(log n): каждый шаг делим данные пополам
n = 1_000_000
steps = 0
while n > 1:
    n //= 2
    steps += 1

print("чтобы поделить миллион пополам до единицы, нужно шагов:", steps)

## 2.4. Как оценить сложность своего кода

Правила простые:

- всё, что вне циклов, - O(1), это можно не считать;
- один цикл по `n` элементам - O(n);
- цикл внутри цикла - сложности **умножаются**;
- два цикла друг за другом - сложности **складываются**, остаётся большая;
- вызов функции внутри цикла - умножаем на сложность этой функции (это главная ловушка).

In [ ]:
def analyze(items):
    count = 0                       # O(1)

    for x in items:                 # O(n)
        count += 1

    for x in items:                 # O(n)   - складывается с предыдущим: O(n) + O(n) = O(n)
        for y in items:             # O(n)   - вложенный, умножается: O(n^2)
            if x == y:
                count += 1

    return count                    # O(1)


# Итого: O(n) + O(n^2) = O(n^2) - главный вклад даёт вложенный цикл
print(analyze([1, 2, 3, 2]))

## 2.5. Худший случай

`x in items` для списка может найти элемент на первом шаге, а может пройти весь список и ничего не найти. Когда говорят про сложность, по умолчанию имеют в виду **худший случай** - то есть O(n).

In [ ]:
data = list(range(200_000))

start = time.perf_counter()
0 in data            # элемент в самом начале - повезло
print(f"первый элемент: {time.perf_counter() - start:.6f} с")

start = time.perf_counter()
-1 in data           # элемента нет - пришлось проверить все
print(f"элемента нет:   {time.perf_counter() - start:.6f} с")

# 3. Сложность операций со списком и строкой

## 3.1. Таблица, которую полезно помнить

| Операция | Сложность | Почему |
|---|---|---|
| `items[i]`, `len(items)` | O(1) | список знает, где лежит каждый элемент |
| `items.append(x)`, `items.pop()` | O(1) | работа только с концом списка |
| `items.insert(0, x)`, `items.pop(0)` | O(n) | все остальные элементы надо сдвинуть |
| `x in items`, `items.count(x)`, `items.index(x)` | O(n) | нужен проход по списку |
| `sum`, `min`, `max` | O(n) | то же самое |
| `items[a:b]` | O(b - a) | срез копирует элементы в новый список |
| `sorted(items)` | O(n log n) | сортировка |

In [ ]:
items = [10, 20, 30, 40, 50]

print(items[2])            # O(1) - сразу
print(len(items))          # O(1)
items.append(60)           # O(1)
items.insert(0, 5)         # O(n) - сдвинули шесть элементов
print(items)
print(25 in items)         # O(n) - проверили все и не нашли

## 3.2. `in` по списку стоит дорого

Один поиск - O(n). Поиск внутри цикла - уже O(n * m), и это самая частая причина медленного кода.

In [ ]:
data = list(range(100_000))
queries = list(range(100_000, 101_000))   # этих значений в списке нет - худший случай

start = time.perf_counter()
found = 0
for q in queries:
    if q in data:          # O(n) внутри цикла по queries -> O(n * m)
        found += 1
print(f"1000 поисков по списку из 100 000: {time.perf_counter() - start:.3f} с")

## 3.3. `insert(0, x)` против `append(x)`

Добавить в конец - O(1). Добавить в начало - O(n): список физически сдвигает все элементы. В цикле это даёт O(n^2) против O(n).

In [ ]:
n = 30_000

start = time.perf_counter()
a = []
for i in range(n):
    a.append(i)            # O(1) каждый раз -> O(n) всего
print(f"append:     {time.perf_counter() - start:.3f} с")

start = time.perf_counter()
b = []
for i in range(n):
    b.insert(0, i)         # O(n) каждый раз -> O(n^2) всего
print(f"insert(0):  {time.perf_counter() - start:.3f} с")

## 3.4. Склейка строк в цикле - тоже ловушка

Строка неизменяема: `s = s + ch` не дописывает символ, а создаёт **новую** строку, скопировав всю старую. В цикле это O(n^2). Правильный способ - собрать части в список и один раз склеить через `join`.

In [ ]:
text = "а" * 100_000

start = time.perf_counter()
s = ""
for ch in text:
    s = s + ch             # каждый раз копируется вся строка -> O(n^2)
print(f"s = s + ch:  {time.perf_counter() - start:.3f} с")

start = time.perf_counter()
parts = []
for ch in text:
    parts.append(ch)       # O(1)
s2 = "".join(parts)        # O(n) один раз
print(f'"".join():   {time.perf_counter() - start:.3f} с')
print("результаты совпали:", s == s2)

## 3.5. Иногда выгодно отсортировать

O(n log n) - это почти O(n), сортировка дешёвая. Если задача «найти два самых близких числа», то вместо перебора всех пар (O(n^2)) можно отсортировать и сравнить только соседей (O(n log n)).

In [ ]:
import random

data = [random.randint(0, 10_000_000) for _ in range(1500)]

start = time.perf_counter()
best = None
for i in range(len(data)):                 # O(n^2): все пары
    for j in range(i + 1, len(data)):
        diff = abs(data[i] - data[j])
        if best is None or diff < best:
            best = diff
print(f"все пары:   {best}   {time.perf_counter() - start:.3f} с")

start = time.perf_counter()
ordered = sorted(data)                     # O(n log n)
best2 = None
for i in range(len(ordered) - 1):          # O(n): только соседи
    diff = ordered[i + 1] - ordered[i]
    if best2 is None or diff < best2:
        best2 = diff
print(f"сортировка: {best2}   {time.perf_counter() - start:.3f} с")

# 4. `set` и `dict`: поиск, который не зависит от размера

Чтобы уйти от O(n) при поиске, нужны структуры, которых мы ещё не проходили. Разберём их здесь - они понадобятся сразу.

## 4.1. `set` - множество

Набор **уникальных** элементов без порядка. Создаётся через `{...}` или `set(список)`. Что умеет:

- `x in s` - проверить наличие, **O(1)**;
- `s.add(x)` - добавить, O(1);
- `s.discard(x)` - удалить, если есть, O(1);
- `len(s)` - размер, O(1).

Пустое множество пишется только как `set()`: `{}` - это пустой словарь.

In [ ]:
marks = [5, 4, 5, 3, 4, 5]

unique = set(marks)
print(unique)                  # порядок может быть любым, дубликаты исчезли
print(len(unique))             # сколько разных оценок

unique.add(2)
unique.discard(3)
print(5 in unique, 3 in unique)

# частый приём: быстро убрать повторы
print(sorted(set(marks)))

## 4.2. `dict` - словарь

Набор пар **ключ - значение**. Ключ как индекс, только не номер, а любое неизменяемое значение.

- `d[k]` - получить значение, O(1), но падает с `KeyError`, если ключа нет;
- `d.get(k, default)` - получить значение или `default`, O(1), не падает;
- `d[k] = v` - записать или перезаписать, O(1);
- `k in d` - проверить **ключ**, O(1);
- `d.items()` - пары, `d.keys()` - ключи, `d.values()` - значения;
- цикл `for k in d` идёт по ключам.

In [ ]:
prices = {"хлеб": 45, "молоко": 89, "сыр": 320}

print(prices["молоко"])                 # 89
print(prices.get("масло"))              # None - ключа нет, но ошибки тоже нет
print(prices.get("масло", 0))           # 0 - значение по умолчанию

prices["масло"] = 180                   # добавили
prices["хлеб"] = 50                     # перезаписали
print("сыр" in prices)                  # проверяется ключ, не значение

for name, price in prices.items():
    print(f"{name}: {price} руб")

## 4.3. Почему это O(1)

Python считает для значения специальное число - **хеш** (`hash(x)`), и по нему сразу вычисляет, в какой ячейке памяти лежит элемент. Поиск = посчитать хеш и заглянуть в одну ячейку. Сколько всего элементов - неважно.

За это приходится платить:

- ключи должны быть **неизменяемыми**: число, строка, кортеж - можно, список - нельзя;
- памяти уходит больше, чем на список с теми же данными.

In [ ]:
print(hash(5))
print(hash("Егор"))
print(hash((1, 2)))        # кортеж можно

try:
    hash([1, 2])           # список нельзя: он меняется, хеш «поплывёт»
except TypeError as e:
    print("ошибка:", e)

## 4.4. Замер: список против множества

Тот же поиск, что в разделе 3.2, но по множеству.

In [ ]:
data_list = list(range(100_000))
data_set = set(data_list)                 # построение множества - один проход, O(n)
queries = list(range(100_000, 101_000))

start = time.perf_counter()
for q in queries:
    q in data_list         # O(n) каждый раз
list_time = time.perf_counter() - start

start = time.perf_counter()
for q in queries:
    q in data_set          # O(1) каждый раз
set_time = time.perf_counter() - start

print(f"список:    {list_time:.4f} с")
print(f"множество: {set_time:.4f} с")
print(f"быстрее в {list_time / set_time:.0f} раз")

Сложность упала с O(n * m) до O(m). Сам `set(data_list)` стоит O(n) - один проход, он окупается с первых же поисков.

## 4.5. Типовой приём: подсчёт через словарь

Посчитать, сколько раз встречается каждый элемент. Через `list.count` в цикле это O(n^2), через словарь - O(n).

In [ ]:
words = ["кот", "пёс", "кот", "ёж", "пёс", "кот"]

# O(n^2): для каждого слова заново пробегаем весь список
slow = {}
for w in words:
    slow[w] = words.count(w)

# O(n): один проход, счётчик берём из словаря
counts = {}
for w in words:
    counts[w] = counts.get(w, 0) + 1

print(slow)
print(counts)
print("совпало:", slow == counts)

## 4.6. Чего `set` и `dict` не умеют

- нет обращения по номеру: `s[0]` для множества - ошибка;
- `set` не хранит порядок и не хранит дубликаты (иногда это именно то, что нужно, а иногда потеря данных);
- ключи только неизменяемые;
- `dict` с Python 3.7 сохраняет порядок добавления ключей, но «третий по счёту» ключ всё равно не получить по индексу.

In [ ]:
s = {10, 20, 30}

try:
    print(s[0])
except TypeError as e:
    print("ошибка:", e)

# если порядок важен - нужен список, множество только для проверки наличия
items = [30, 10, 30, 20, 10]
seen = set()
result = []
for x in items:
    if x not in seen:      # O(1)
        seen.add(x)
        result.append(x)
print(result)              # дубликаты убраны, порядок сохранён, всего O(n)

# 5. Пространственная сложность

## 5.1. Что считаем

Пространственная сложность - сколько **дополнительной** памяти заводит код, не считая самих входных данных. Записывается так же:

- O(1) - несколько переменных, сколько бы ни было данных;
- O(n) - новая структура размером со входные данные;
- O(n^2) - например, список всех пар.

Один и тот же результат можно получить за O(1) и за O(n) памяти.

In [ ]:
def sum_positive_list(numbers):
    positive = []                  # O(n) памяти: в худшем случае копия всего списка
    for n in numbers:
        if n > 0:
            positive.append(n)
    return sum(positive)


def sum_positive_counter(numbers):
    total = 0                      # O(1) памяти: одна переменная
    for n in numbers:
        if n > 0:
            total += n
    return total


data = [3, -1, 7, 0, 5]
print(sum_positive_list(data), sum_positive_counter(data))

## 5.2. Измеряем размер объекта

`sys.getsizeof(x)` возвращает размер объекта в байтах. Для списка он показывает размер самого списка (ссылок), без размера элементов, но для сравнения этого достаточно.

In [ ]:
import sys

for n in [0, 10, 1000, 100_000]:
    items = list(range(n))
    print(f"список из {n:>7} элементов: {sys.getsizeof(items):>9} байт")

## 5.3. Лень как экономия памяти

`range`, `enumerate`, `zip` не создают список. Они выдают элементы по одному, поэтому занимают O(1) памяти независимо от `n`. Именно поэтому `range(10 ** 9)` создаётся мгновенно, а `list(range(10 ** 9))` положит компьютер.

In [ ]:
print("range(10**9):        ", sys.getsizeof(range(10 ** 9)), "байт")
print("list(range(100_000)):", sys.getsizeof(list(range(100_000))), "байт")

pairs = zip([1, 2, 3], "abc")
print("zip:                 ", sys.getsizeof(pairs), "байт")

# пока по нему не пройдёшь, элементов в памяти нет
print(list(pairs))

## 5.4. `tracemalloc`: сколько памяти съела функция

`sys.getsizeof` смотрит на один объект. Чтобы увидеть, сколько памяти заняла работа целиком, есть модуль `tracemalloc`: он показывает пиковое потребление между `start()` и `get_traced_memory()`.

In [ ]:
import tracemalloc

def measure(func, data):
    tracemalloc.start()
    func(data)
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    return peak


data = list(range(200_000))
print(f"через список:   {measure(sum_positive_list, data):>9} байт")
print(f"через счётчик:  {measure(sum_positive_counter, data):>9} байт")

Результат один и тот же, а памяти первый вариант требует в тысячи раз больше. На больших данных это разница между «работает» и «не хватило памяти».

## 5.5. Время против памяти

Почти всегда ускорение покупается памятью, и наоборот:

- `set(items)` - потратили O(n) памяти, получили поиск за O(1) вместо O(n);
- счётчик вместо списка - сэкономили O(n) памяти, но данные можно пройти только один раз;
- сохранить посчитанное, чтобы не считать заново, - заплатили памятью за время.

Что выбрать, зависит от того, чего не хватает. Если данные маленькие - не выбирайте ничего, пишите просто и понятно.

In [ ]:
names = ["Егор", "Аня", "Пётр", "Лиза"]
banned = ["Пётр", "Олег"]

# O(n * m) времени, O(1) лишней памяти
allowed_slow = [name for name in names if name not in banned]

# O(n + m) времени, O(m) лишней памяти на множество
banned_set = set(banned)
allowed_fast = [name for name in names if name not in banned_set]

print(allowed_slow)
print(allowed_fast)

# Частые ошибки по этой теме

- Считать, что O-нотация про секунды. Она про рост: O(n^2) код на 10 элементах быстрее O(n) кода, разница появляется на больших данных.
- Оптимизировать то, что не замерял. Сначала замер, потом вывод.
- `x in items` внутри цикла по списку - незаметный O(n^2).
- `s = s + ch` в цикле вместо `"".join(parts)`.
- `items.insert(0, x)` или `items.pop(0)` в цикле вместо `append` / `pop`.
- `items.count(x)` внутри цикла по тем же `items` - O(n^2) там, где словарь даёт O(n).
- Думать, что вызов функции бесплатный: `sorted` внутри цикла - это O(n^2 log n).
- Использовать `set` там, где важен порядок или нужны дубликаты.
- Строить список целиком, когда достаточно счётчика: результат тот же, памяти в тысячи раз больше.

# Памятка

**Сложность по времени** - как растёт число шагов при росте `n`. **Сложность по памяти** - сколько лишней памяти при этом нужно.

Классы по возрастанию: O(1) < O(log n) < O(n) < O(n log n) < O(n^2).

Как оценить код: считаем только циклы, вложенные - умножаем, последовательные - складываем и оставляем больший, константы отбрасываем.

Дорогие операции со списком: `in`, `count`, `index`, `insert(0)`, `pop(0)` - все O(n).

`set` и `dict` дают `in` за O(1) ценой лишней памяти; ключи обязаны быть неизменяемыми.

Подсчёт частот - словарь и `get(k, 0) + 1`, а не `count` в цикле.

Убрать дубликаты: `set(items)` без порядка, `set` + список - с порядком.

`range`, `enumerate`, `zip` ленивые: O(1) памяти вместо O(n).

Замер: `time.perf_counter()` для времени, `sys.getsizeof` для объекта, `tracemalloc` для функции.

Правило: сначала понятный код, потом замер, и только потом оптимизация.